In [16]:
%pip install fake-useragent

   ---------------------------------------- 0.0/161.7 kB ? eta -:--:--
   -- ------------------------------------- 10.2/161.7 kB ? eta -:--:--
   ------- ------------------------------- 30.7/161.7 kB 435.7 kB/s eta 0:00:01
   ------------------------------------- -- 153.6/161.7 kB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 161.7/161.7 kB 1.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import random
from web.config.database import SessionLocal
import OpenDartReader
import requests
import re
from bs4 import BeautifulSoup
from sqlalchemy import text
import time  # [추가] 딜레이를 위해 추가
import OpenDartReader
from dotenv import find_dotenv, load_dotenv
from fake_useragent import UserAgent # [추가]
import os

# UserAgent 객체 생성 (캐싱을 위해 루프 밖에서 한 번만 생성하거나 필요시 생성)
ua = UserAgent()

load_dotenv(find_dotenv())

dart = OpenDartReader(os.getenv("dart_api_key"))

target_idx = {
    "company_overview": "회사의 개요",
    "business_contents": "사업의 내용",
    "shareholder_info": "주주에 관한 사항",
    "investor_protection": "그 밖에 투자자 보호를 위하여 필요한 사항",
    "contingent_liabilities": "우발부채 등에 관한 사항"
}

# [수정] 1. 이미 parsed_disclosure_file에 존재하는(작업 완료된) 건은 제외하고 조회
# LEFT JOIN ... WHERE ... IS NULL 방식을 사용하여 아직 작업 안 된 것만 가져옵니다.
sql_select = text("""
    SELECT df.id, dl.company_id, dl.rcept_no
    FROM disclosure_file df
    JOIN disclosure_list dl ON df.disclosure_id = dl.id
    LEFT JOIN parsed_disclosure_file pdf ON df.id = pdf.disclosure_file_id
    WHERE pdf.disclosure_file_id IS NULL
    ORDER BY dl.company_id asc
""")

session = SessionLocal()
try:
    results = session.execute(sql_select).all()
    print(f"총 {len(results)}개의 미처리 작업을 시작합니다.")

    for id, company_id, recept_no in results:
        print(f"Processing ID: {id}, RCP_NO: {recept_no}")
        rcp_no = recept_no

        try:
            # [팁] 차단 방지를 위해 대기 시간도 랜덤으로 설정 (1초 ~ 3초 사이)
            time.sleep(random.uniform(1, 3))
            sub_docs = dart.sub_docs(rcp_no)
        except Exception as e:
            print(f"DART API 호출 중 오류 발생: {e}")
            continue

        row_data = {
            "disclosure_file_id": id,
            "company_id": company_id,
            "company_overview": None,
            "business_contents": None,
            "shareholder_info": None,
            "investor_protection": None,
            "contingent_liabilities": None
        }

        if sub_docs is not None:
            for key, value in target_idx.items():
                target_doc = sub_docs[sub_docs['title'].str.contains(value)]

                if not target_doc.empty:
                    try:
                        target_url = target_doc['url'].values[0]

                        # [핵심] 랜덤 User-Agent 헤더 생성
                        headers = {
                            'User-Agent': ua.random,
                            'Referer': 'https://dart.fss.or.kr/' # 리퍼러를 추가하면 더 사람처럼 보입니다.
                        }

                        # headers 파라미터 추가
                        response = requests.get(target_url, headers=headers)

                        time.sleep(random.uniform(0.1, 1.0)) # 크롤링 간격도 랜덤화

                        soup = BeautifulSoup(response.content, 'html.parser')
                        raw_text = soup.get_text(separator='\n').replace('\xa0', ' ')
                        clean_text = re.sub(r'\n+', '\n', raw_text).strip()

                        row_data[key] = clean_text

                    except Exception as e:
                        print(f" > {value} 파싱 에러: {e}")
                        row_data[key] = None
                else:
                    row_data[key] = None

        # 4. 저장
        try:
            sql_insert = text("""
                INSERT INTO parsed_disclosure_file (
                    disclosure_file_id,
                    company_id,
                    company_overview,
                    business_contents,
                    shareholder_info,
                    investor_protection,
                    contingent_liabilities,
                    created_at
                ) VALUES (
                    :disclosure_file_id,
                    :company_id,
                    :company_overview,
                    :business_contents,
                    :shareholder_info,
                    :investor_protection,
                    :contingent_liabilities,
                    NOW()
                )
            """)

            session.execute(sql_insert, row_data)
            session.commit()
            print(f"== ID {id} 저장 완료 (남은 작업: {len(results) - results.index((id, company_id, recept_no)) - 1}개) ==\n")

        except Exception as e:
            session.rollback()
            print(f"DB 저장 중 에러 (ID: {id}): {e}")

except Exception as e:
    print(f"치명적 오류 발생: {e}")
finally:
    session.close()

총 3447개의 미처리 작업을 시작합니다.
Processing ID: 916, RCP_NO: 20241114001621
== ID 916 저장 완료 (남은 작업: 3446개) ==

Processing ID: 918, RCP_NO: 20240516001968
== ID 918 저장 완료 (남은 작업: 3445개) ==

Processing ID: 920, RCP_NO: 20241114002482
